In [ ]:
from importlib import reload
import plotting_utils as pu

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

freq = 32

cell_types = ["Exc", "Pvalb", "Sst", "Vip", "Htr3a"]
colors = [
    "tab:red",
    "tab:blue",
    "tab:green",
    "tab:purple",
    "tab:purple",
]
layers = ["L1", "L2/3", "L4", "L5", "L6"]

cell_labels = {
    "Exc": "Exc",
    "Pvalb": "Pvalb",
    "Sst": "Sst",
    "Vip": "Vip",
    "Htr3a": "L1 Inh",
}

In [ ]:
net_name = "core"  # change to "core_<variant>" for a model variant
pu.plot_raster_movie(
    config_file=f"{net_name}/configs/natural_scenes_10trials_checkpoint/config_0.json",
    freq=freq,
    cell_types=cell_types,
    cell_labels=cell_labels,
    color_order=colors,
    bins=np.linspace(0, 3001, 301),
    n_xticks=6,
    ratio=[1, 20],
    s=0.3,
    legend=True,
    figsize=(5, 5),
    save_path="paper_figs/fig2C_nat_scenes_raster_Jul17_2026.tiff",
)
# fig.savefig("raster_drifting_gratings_alldirs.pdf", dpi=600, bbox_inches="tight")

In [ ]:
pu.plot_raster_movie(
    config_file=f"{net_name}/configs/natural_movie_one_10trials_checkpoint/config_0.json",
    freq=freq,
    cell_types=cell_types,
    cell_labels=cell_labels,
    color_order=colors,
    bins=np.linspace(0, 3001, 301),
    n_xticks=12,
    ratio=[1, 20],
    s=0.3,
    legend=True,
    figsize=(5, 5),
    save_path="paper_figs/fig2D_nat_movie_raster_Jul17_2026.tiff",
)
# fig.savefig("raster_drifting_gratings_alldirs.pdf", dpi=600, bbox_inches="tight")

## Fig 2 suppl: firing rates distributions

In [ ]:
Perturb_dict = {
    "drifting gratings": "8dir_10trials_checkpoint",
    "natural scenes": "natural_scenes_10trials_checkpoint",
    "natural movie one": "natural_movie_one_10trials_checkpoint",
    "natural movie two": "natural_movie_two_10trials_checkpoint",
}

config_files_path = f"{net_name}/configs"

In [ ]:

# Generate plasma colors for the stimulus conditions
stim_colors = np.delete(plt.colormaps["plasma"](np.linspace(0, 1, 7)), 6, axis=0)
stim_palette = {cond: stim_colors[i] for i, cond in enumerate(Perturb_dict.keys())}

v1df_all = pd.DataFrame()
for condition, file_info in Perturb_dict.items():
    print(condition)
    file_name = config_files_path + f"/{file_info}" + "/config_0.json"
    v1df, spike_df = pu.data_organize(
        file_name, t_min=500, t_max=3000, radius=pu.settings["core"]["radius"]
    )
    v1df["condition"] = condition
    v1df_all = pd.concat([v1df_all, v1df])

v1df_all["cell_type"] = v1df_all["cell_type"].map(cell_labels)

v1df_all["cell_layer"] = v1df_all.apply(
    lambda row: row["layer"] + " " + row["cell_type"] if row["layer"] != "L1" else "L1 Inh", axis=1
)

fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
fig.patch.set_facecolor("none")
sns.boxplot(
    ax=ax,
    data=v1df_all,
    x="cell_layer",
    y="fr_stimulus",
    hue="condition",
    hue_order=list(Perturb_dict.keys()),
    palette=stim_palette,
    flierprops=dict(marker="x", markeredgecolor="black", markersize=2, zorder=2),
    medianprops=dict(color="tab:brown", alpha=0.7, zorder=2),
    boxprops=dict(linewidth=0.5, zorder=2),
    capprops=dict(color="black", linewidth=0.5, zorder=2),
    order=v1df_all["cell_layer"].unique().tolist(),
)
plt.legend(fontsize="x-small")
ax.set_xticklabels([t.get_text().replace("L1 Htr3a", "L1 Inh") for t in ax.get_xticklabels()], rotation=45)
plt.xlabel(None)
plt.ylabel("Firing rate [spk/s]")

# plot background layer divider
layer_sep = np.cumsum(v1df_all.groupby("layer")["cell_type"].nunique().values) - 0.5
for i in range(1, len(layer_sep), 2):
    ax.axvspan(layer_sep[i - 1], layer_sep[i], color="gray", alpha=0.1)
# plot layer text
xlim = ax.get_xlim()
ylim = ax.get_ylim()
layer_sep = np.insert(layer_sep, 0, xlim[0])
layer_sep[-1] = xlim[1]
layers_list = ["L1", "L2/3", "L4", "L5", "L6"]
for i in range(1, len(layer_sep)):
    plt.text(
        (layer_sep[i - 1] + layer_sep[i]) / 2 - 0.3,
        ylim[1] * 0.8,
        layers_list[i - 1],
        fontsize=12,
        weight="bold",
    )

fig.tight_layout()
fig.savefig(
    "paper_figs/fig2_suppl_fr_dist_stim_conditions_plasma_Jul17_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
